# Rozwiązania — lekcja 7: Analiza z mniejszą liczbą podpowiedzi

To przykładowe odpowiedzi. Inny poprawny sposób jest równie dobry. Każde rozwiązanie uruchom po komórce przygotowującej dane poniżej.


In [ ]:
from pathlib import Path

# Notebook działa po uruchomieniu z katalogu kursu lub z jego podkatalogu.
KATALOG_KURSU = next(
    katalog for katalog in [Path.cwd(), *Path.cwd().parents]
    if (katalog / "dane" / "sprzedaz_kawiarni.csv").exists()
)
DANE = KATALOG_KURSU / "dane"
WYNIKI = KATALOG_KURSU / "wyniki"
WYNIKI.mkdir(exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
biblioteka = pd.read_csv(DANE / "wypozyczenia_biblioteki.csv")


## Zbadaj dane

Sprawdź początek, rozmiar, typy, braki i powtórzone ID. Zapisz, co opisuje jeden wiersz i które kolumny przydadzą się do odpowiedzi.


In [ ]:
display(biblioteka.head())
print(biblioteka.shape)
biblioteka.info()
print(biblioteka.isna().sum())
print("Powtórzone ID:", biblioteka.duplicated(subset="id_wypozyczenia").sum())


**Interpretacja / kontrola:** **180 wierszy**, **6 kolumn**, **30 braków daty zwrotu**, brak powtórzonych ID. Pustych zwrotów nie usuwamy z całej analizy.


## Najczęściej wypożyczane kategorie

Policz wypożyczenia według kategorii i narysuj wykres. Uwzględnij również jeszcze niezwrócone książki.


In [ ]:
liczby = biblioteka["kategoria"].value_counts().sort_values()
print(liczby)
ax = liczby.plot.barh(figsize=(7, 4))
ax.set(title="Wypożyczenia według kategorii", xlabel="Liczba wypożyczeń", ylabel="Kategoria")
plt.tight_layout()
plt.show()


**Interpretacja / kontrola:** Suma kategorii to **180**. To liczba wypożyczeń, a nie liczba unikalnych czytelników ani różnych książek.


## Czas zakończonych wypożyczeń

Wybierz zakończone wypożyczenia. Porównaj średnią i medianę czasu według kategorii oraz podaj liczebności. Mediana to wartość środkowa uporządkowanych wyników.


In [ ]:
biblioteka["data_wypozyczenia"] = pd.to_datetime(biblioteka["data_wypozyczenia"])
biblioteka["data_zwrotu"] = pd.to_datetime(biblioteka["data_zwrotu"])
biblioteka["dni_wypozyczenia"] = (biblioteka["data_zwrotu"] - biblioteka["data_wypozyczenia"]).dt.days
zakonczone = biblioteka[biblioteka["data_zwrotu"].notna()].copy()
tabela = zakonczone.groupby("kategoria").agg(
    liczba=("dni_wypozyczenia", "count"),
    srednia_dni=("dni_wypozyczenia", "mean"),
    mediana_dni=("dni_wypozyczenia", "median"),
)
print(tabela.round(1))
print("Zakończone:", len(zakonczone))


**Interpretacja / kontrola:** Uwzględniasz **150 zakończonych wypożyczeń**. Nie ma ujemnych czasów. Wynik nie opisuje długości wszystkich wypożyczeń, bo 30 nadal trwa.


## Wypożyczenia w czasie

Policz wypożyczenia według miesiąca rozpoczęcia i przygotuj wykres liniowy. Zapisz jedno ograniczenie analizy.


In [ ]:
biblioteka["data_wypozyczenia"] = pd.to_datetime(biblioteka["data_wypozyczenia"])
biblioteka["miesiac"] = biblioteka["data_wypozyczenia"].dt.month
miesiecznie = biblioteka.groupby("miesiac").size().sort_index()
print(miesiecznie)
ax = miesiecznie.plot.line(marker="o", figsize=(7, 4))
ax.set(title="Wypożyczenia według miesiąca rozpoczęcia — 2026", xlabel="Miesiąc", ylabel="Liczba wypożyczeń", xticks=miesiecznie.index)
plt.tight_layout()
plt.show()


**Interpretacja / kontrola:** Wyniki sumują się do **180**. Zbiór obejmuje styczeń–maj; brak rozpoczęć w czerwcu wynika z konstrukcji danych. Nie traktuj go jako dowodu spadku zainteresowania.
